# Week 3 · ETA Regression Baseline: mean/median floor vs three regressors

# Requirements: pip install scikit-learn pandas numpy
# Uses: the `zoro` package (rebuilds features deterministically; no GPU, no API key).

We predict **`delay_hours`** (actual minus planned arrival). An operations team reads
`planned_arrival + predicted_delay` as the adjusted ETA, so predicting the delay *is*
predicting the ETA correction. The discipline: a time-aware split, a **baseline floor**
(mean/median), three scikit-learn regressors, and MAE/RMSE, with the metric chosen
*before* any model is fit.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
p = pathlib.Path.cwd()
while not (p / "zoro").is_dir() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
from zoro import data
print("loaded zoro.data")

### The feature table

The signal for lateness lives across three tables: lane **distance**, carrier
**reliability** (`on_time_rate`), and the shipment's own **weight/value** plus
**weather** and calendar features. This mirrors the Week 2 silver cleaning, impute the
planted NaNs, drop duplicates, so the notebook runs standalone.

In [ ]:
# Build the feature table: join shipments with lane distance and carrier profile,
# impute the planted NaNs (mirrors Week 2 cleaning), and derive calendar features.
def build_features(n=100_000, seed=42):
    ships = data.shipments(n, seed=seed)
    lanes = data.lanes(20, seed=11)
    carriers = data.carriers(20, seed=7)
    df = (ships
          .merge(lanes[["lane_id", "distance_km"]], on="lane_id", how="left")
          .merge(carriers[["carrier_id", "on_time_rate", "fleet_size", "region"]], on="carrier_id", how="left"))
    df["weight_kg"] = df["weight_kg"].fillna(df.groupby("commodity")["weight_kg"].transform("median"))
    df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())
    df = df.drop_duplicates().reset_index(drop=True)
    df["month"] = df["planned_departure"].dt.month
    df["day_of_week"] = df["planned_departure"].dt.dayofweek
    return df

df = build_features(100_000, seed=42)
print("feature table:", df.shape)

NUM_COLS = ["distance_km", "weight_kg", "value_usd", "on_time_rate", "fleet_size", "month", "day_of_week"]
CAT_COLS = ["weather_severity", "commodity"]

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
prep = ColumnTransformer([
    ("num", StandardScaler(), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
])
print("features: numeric =", NUM_COLS, "| categorical =", CAT_COLS)

### The time-aware split: why we never shuffle

Shipment data is a time series. Randomly shuffling would let a "future" shipment (with
its weather and holiday congestion) leak into training while a "past" one sits in test
the model learns the future and looks brilliant on the past. That is **leakage**. We
split chronologically: train on the earliest months, validate on the next window, test
on the most recent.

In [ ]:
# Time-aware split: chronological, never shuffled. Data spans 2025-01-01 .. 2025-12-30.
CUT1 = "2025-08-01"
CUT2 = "2025-10-01"
train_df = df[df["planned_departure"] < CUT1].reset_index(drop=True)
val_df   = df[(df["planned_departure"] >= CUT1) & (df["planned_departure"] < CUT2)].reset_index(drop=True)
test_df  = df[df["planned_departure"] >= CUT2].reset_index(drop=True)
print("train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

### The baseline floor

Before any model, compute the obvious answer: predict the training mean (or median)
delay for everything. A real model must *beat this and state the delta*, otherwise it
is polishing a worse-than-nothing answer.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_test = test_df["delay_hours"]
mean_pred = train_df["delay_hours"].mean()
median_pred = train_df["delay_hours"].median()

for label, pred in [("mean", mean_pred), ("median", median_pred)]:
    mae = mean_absolute_error(y_test, np.full(len(y_test), pred))
    rmse = np.sqrt(mean_squared_error(y_test, np.full(len(y_test), pred)))
    print(f"baseline {label:<6} MAE={mae:.3f}  RMSE={rmse:.3f}")

### Three regressors

A linear model, a random forest, and gradient boosting, the standard tabular ladder.
Each wraps the same `ColumnTransformer` (scale the numerics, one-hot the categoricals)
so the comparison is fair. Trees don't *need* scaling, but the pipeline keeps every
model on identical inputs. Fitting the ensembles may take a minute or two on CPU.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=0),
}

results = {}
fitted = {}
test_preds = {}
for name, model in models.items():
    pipe = Pipeline([("prep", prep), ("model", model)])
    pipe.fit(train_df, train_df["delay_hours"])
    preds = pipe.predict(test_df)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results[name] = {"mae": mae, "rmse": rmse}
    fitted[name] = pipe
    test_preds[name] = preds
    print(f"{name:<18} MAE={mae:.3f}  RMSE={rmse:.3f}")

comp = pd.DataFrame(results).T.round(3)
print()
print(comp.to_string())

### Compare, and slice the winner's errors

Pick the best model by test MAE, then slice its errors by **weather severity**, a
first error analysis. RMSE far above MAE means a few shipments are dramatically wrong;
severe weather is the natural first suspect.

In [ ]:
best_name = comp["mae"].idxmin()
print("best model by test MAE:", best_name)

test_meta = test_df[["weather_severity", "carrier_id", "lane_id"]].reset_index(drop=True).copy()
test_meta["err"] = np.abs(test_preds[best_name] - y_test.to_numpy())
print()
print("MAE by weather severity (test):")
print(test_meta.groupby("weather_severity")["err"].mean().round(3).to_string())

### The metric

One number to close: the best model's test MAE, in hours. This is the number Week 4's
neural net must beat.

In [ ]:
print("BEST_TEST_MAE:", round(float(comp.loc[best_name, "mae"]), 3))
print("BEST_TEST_RMSE:", round(float(comp.loc[best_name, "rmse"]), 3))